## Verify Parseval theorem

In [1]:
import numpy as np

def parseval_default_norm(x: np.ndarray):
    """
    Parseval check for numpy's default FFT normalization:
    forward FFT unscaled, inverse scaled by 1/N.
    """
    N = x.size
    X = np.fft.fft(x)  # default norm
    time_energy = np.sum(np.abs(x)**2)
    freq_energy = (1.0 / N) * np.sum(np.abs(X)**2)
    return time_energy, freq_energy, float(np.abs(time_energy - freq_energy))

def parseval_unitary_norm(x: np.ndarray):
    """
    Parseval check for unitary FFT normalization:
    both forward and inverse scaled by 1/sqrt(N).
    """
    X = np.fft.fft(x, norm="ortho")
    time_energy = np.sum(np.abs(x)**2)
    freq_energy = np.sum(np.abs(X)**2)
    return time_energy, freq_energy, float(np.abs(time_energy - freq_energy))

def demo():
    rng = np.random.default_rng(0)

    tests = {}

    # 1) Real random signal
    x1 = rng.standard_normal(1024)*10
    tests["real_random"] = x1

    # 2) Complex random signal
    x2 = rng.standard_normal(1024) + 1j * rng.standard_normal(1024)
    tests["complex_random"] = x2

    # 3) Single sinusoid (real)
    N = 1024
    n = np.arange(N)
    f0 = 37  # bins
    x3 = np.cos(2 * np.pi * f0 * n / N)
    tests["single_cosine"] = x3

    # 4) Kronecker delta (impulse)
    x4 = np.zeros(N, dtype=complex)
    x4[0] = 1.0
    tests["impulse"] = x4

    # 5) Rectangular pulse
    x5 = np.zeros(N)
    x5[:64] = 1.0
    tests["rect_pulse_64"] = x5

    print("=== Parseval verification (numpy default FFT) ===")
    for name, x in tests.items():
        te, fe, err = parseval_default_norm(np.asarray(x))
        print(f"{name:>16s} | time={te:.6f} freq={fe:.6f} abs error={err:.3e}")

    print("\n=== Parseval verification (unitary FFT, norm='ortho') ===")
    for name, x in tests.items():
        te, fe, err = parseval_unitary_norm(np.asarray(x))
        print(f"{name:>16s} | time={te:.6f} freq={fe:.6f} abs error={err:.3e}")

if __name__ == "__main__":
    demo()

=== Parseval verification (numpy default FFT) ===
     real_random | time=96966.646862 freq=96966.646862 abs error=1.455e-11
  complex_random | time=2082.128596 freq=2082.128596 abs error=4.547e-13
   single_cosine | time=512.000000 freq=512.000000 abs error=2.274e-13
         impulse | time=1.000000 freq=1.000000 abs error=0.000e+00
   rect_pulse_64 | time=64.000000 freq=64.000000 abs error=7.105e-15

=== Parseval verification (unitary FFT, norm='ortho') ===
     real_random | time=96966.646862 freq=96966.646862 abs error=1.455e-11
  complex_random | time=2082.128596 freq=2082.128596 abs error=4.547e-13
   single_cosine | time=512.000000 freq=512.000000 abs error=2.274e-13
         impulse | time=1.000000 freq=1.000000 abs error=0.000e+00
   rect_pulse_64 | time=64.000000 freq=64.000000 abs error=7.105e-15


## AMSE in numpy

In [2]:
import numpy as np

def unitary_fft(x: np.ndarray) -> np.ndarray:
    """DFT with unitary normalization (1/sqrt(N) in both directions)."""
    return np.fft.fft(x, norm="ortho")

def mse_time(x: np.ndarray, y: np.ndarray) -> float:
    """Time-domain mean squared error."""
    e = x - y
    return float(np.mean(np.abs(e) ** 2))

def mse_freq_unitary(x: np.ndarray, y: np.ndarray):
    """
    Frequency-domain MSE using the unitary FFT auto–cross view:

      R_xx = (1/N) * sum_k |X[k]|^2
      R_yy = (1/N) * sum_k |Y[k]|^2
      R_xy = (1/N) * sum_k X[k] * conj(Y[k])

      MSE = R_xx + R_yy - 2 * Re{R_xy}
    """
    N = x.size
    X = unitary_fft(x)
    Y = unitary_fft(y)

    R_xx = (1.0 / N) * np.sum(np.abs(X) ** 2)
    R_yy = (1.0 / N) * np.sum(np.abs(Y) ** 2)
    R_xy = (1.0 / N) * np.sum(X * np.conj(Y))

    mse_freq = float(R_xx + R_yy - 2.0 * np.real(R_xy))
    return float(R_xx), float(R_yy), complex(R_xy), mse_freq

def amse_freq_unitary(x: np.ndarray, y: np.ndarray, eps: float = 1e-12) -> float:
    r"""
    AMSE(x,y) = sum_k ( sqrt(PSD_k(x)) - sqrt(PSD_k(y)) )^2
                + 2 * max(PSD_k(x), PSD_k(y)) * (1 - Coh_k(x,y))

    With unitary-FFT per-bin definitions:
        PSD_k(x) = (1/N) * |X~[k]|^2
        PSD_k(y) = (1/N) * |Y~[k]|^2
        Coh_k(x,y) = cos(Δφ_k) = Re{X~[k] Y~*[k]} / (|X~[k]| |Y~[k]| + eps)

    Notes:
      * This "coherence" equals the cosine of the phase difference, which
        keeps AMSE close to the unitary MSE per-bin.
      * If you instead want magnitude or mag-squared coherence, replace Coh_k
        by |X~Y~*| / (|X~||Y~|) or its square.
    """
    x = np.asarray(x); y = np.asarray(y)
    if x.shape != y.shape: raise ValueError("x,y must have the same shape")
    N = x.size
    if N == 0: return 0.0

    X = unitary_fft(x)
    Y = unitary_fft(y)

    # Per-bin PSDs that sum to average power
    PSDx = (np.abs(X)**2) / N
    PSDy = (np.abs(Y)**2) / N

    # Per-bin "coherence" chosen as cos of phase difference (real coherency)
    num = np.real(X * np.conj(Y))                      # Re{X Y*}
    den = (np.abs(X) * np.abs(Y)) + eps               # |X||Y|
    Coh = num / den                                   # in [-1, 1]

    # AMSE per bin and total
    term_amp   = (np.sqrt(PSDx) - np.sqrt(PSDy))**2
    term_phase = 2.0 * np.maximum(PSDx, PSDy) * (1.0 - Coh)
    AMSE_bins  = term_amp + term_phase
    return float(np.sum(AMSE_bins))


def demo():
    rng = np.random.default_rng(0)
    N = 1024
    n = np.arange(N)

    cases = []

    # 1) Identical real signals (MSE ~ 0)
    x = rng.standard_normal(N)*100+0.2
    y = x.copy()
    cases.append(("identical_real", x, y))

    # 2) y = x + small noise (real)
    x = rng.standard_normal(N)
    y = x + 0.1 * rng.standard_normal(N)
    cases.append(("noisy_real_sigma0.1", x, y))

    # 3) Complex sinusoid with phase shift
    f0 = 37
    x = np.exp(1j * 2 * np.pi * f0 * n / N)
    y = np.exp(1j * (2 * np.pi * f0 * n / N + np.pi / 4))  # 45° shift
    cases.append(("complex_sinusoid_phase_shift", x, y))

    # 4) Linear relation + complex noise
    x = rng.standard_normal(N) + 1j * rng.standard_normal(N)
    y = 0.6 * x + 0.3 * (rng.standard_normal(N) + 1j * rng.standard_normal(N))
    cases.append(("linear_relation_plus_noise", x, y))

    print("Case                                MSE_time       MSE_freq        AMSE_freq   ")
    print("-" * 80)
    for name, x, y in cases:
        mse_t = mse_time(x, y)
        _, _, _, mse_f = mse_freq_unitary(x, y)
        amse = amse_freq_unitary(x, y)
        print(f"{name:30s}  {mse_t:12.6f}    {mse_f:12.6f}   {amse:12.6f}  ")

if __name__ == "__main__":
    demo()

Case                                MSE_time       MSE_freq        AMSE_freq   
--------------------------------------------------------------------------------
identical_real                      0.000000        0.000000       0.000000  
noisy_real_sigma0.1                 0.009730        0.009730       0.010217  
complex_sinusoid_phase_shift        0.585786        0.585786       0.585786  
linear_relation_plus_noise          0.516745        0.516745       0.645580  


## AMSE in torch

In [3]:
import torch
import torch.nn as nn
from torch import Tensor


def _unitary_fft(x: Tensor, dim: int = -1) -> Tensor:
    """Unitary FFT (norm='ortho') along `dim`."""
    return torch.fft.fft(x, dim=dim, norm="ortho")

def _per_bin_psd_unitary(X: Tensor, N: int) -> Tensor:
    """Per-bin PSD so that sum_k PSD_k = average power per sample."""
    return (X.abs()**2) / float(N)

def _per_bin_cos_phase(X: Tensor, Y: Tensor, eps: float) -> Tensor:
    """
    cos(Δφ_k) = Re{X Y*} / (|X||Y| + eps), elementwise over the FFT axis.
    Returns a real tensor in [-1, 1].
    """
    num = torch.real(X * torch.conj(Y))
    den = X.abs() * Y.abs() + eps
    cphi = num / den
    return torch.clamp(cphi, -1.0, 1.0)

class AMSELoss(nn.Module):
    """
    AMSE loss under unitary DFT.
    AMSE = sum_k (sqrt(PSD_x)-sqrt(PSD_y))^2
                + 2*max(PSD_x,PSD_y)*(1 - cosΔφ)

    Args:
        fft_dim:  axis to transform (default: last)
        reduction: 'mean' | 'sum' | 'none'
        eps: small constant for numerical stability in divisions/sqrt
    """
    def __init__(
        self,
        fft_dim: int = -1,
        reduction: str = "mean",
        eps: float = 1e-12,
    ):
        super().__init__()

        self.fft_dim = fft_dim
        self.reduction = reduction
        self.eps = eps

    def forward(self, pred: Tensor, target: Tensor) -> Tensor:
        """
        pred, target: (..., N) real or complex tensors.
        The FFT is applied along `fft_dim`. All leading dims are batch-like.
        """
        if pred.shape != target.shape:
            raise ValueError("pred and target must have identical shapes")

        N = pred.size(self.fft_dim)
        # Unitary FFT
        X = _unitary_fft(pred, dim=self.fft_dim)
        Y = _unitary_fft(target, dim=self.fft_dim)

        # Per-bin PSDs
        PSDx = _per_bin_psd_unitary(X, N)          # (..., N)
        PSDy = _per_bin_psd_unitary(Y, N)          # (..., N)

        # cos(Δφ_k) per bin
        cphi = _per_bin_cos_phase(X, Y, self.eps)  # real (..., N)
        # Amplitude term
        term_amp = (torch.sqrt(PSDx + self.eps) - torch.sqrt(PSDy + self.eps))**2
        # Phase/coherence-like term
        term_phase = 2.0 * torch.maximum(PSDx, PSDy) * (1.0 - cphi)

        # Sum over frequency bins
        amse_per_example = (term_amp + term_phase).sum(dim=self.fft_dim)

        # Reduce
        if self.reduction == "mean":
            return amse_per_example.mean()
        elif self.reduction == "sum":
            return amse_per_example.sum()
        else:
            return amse_per_example  # shape: batch-like dims only
        
def mse_time(x: Tensor, y: Tensor) -> Tensor:
    """Time-domain mean squared error."""
    e = x - y
    return torch.mean(torch.abs(e) ** 2, dim=-1)

def mse_freq_unitary_torch(a: Tensor, b: Tensor) -> Tensor:
    A = _unitary_fft(a, dim=-1)
    B = _unitary_fft(b, dim=-1)
    return ((A - B).abs()**2).sum(dim=-1) / a.size(-1)


# -------------------- Tiny sanity check --------------------
if __name__ == "__main__":
    torch.manual_seed(0)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    N = 2048
    n = torch.arange(N, device=device)

    # batch=3 example, real signals
    f0 = 73
    x = torch.cos(2*torch.pi*f0*n/N)[None, :]                # (1,N)

    # Identical targets:
    y_1 =x.clone()

    # phase-shifted
    y_2 = torch.cos(2*torch.pi*f0*n/N + 0.3)[None, :]          

    # Linear relation + complex noise
    y_3 = 0.6 * x + 0.3 * (torch.randn_like(x) + 1j * torch.randn_like(x))

    #  y = x + small noise (real)
    y_4 = x + 0.1 * torch.randn_like(x)


    amse_loss = AMSELoss(reduction="none").to(device)
    print("Case                                 MSE_time       MSE_freq      AMSE_freq  ")
    print("-" * 85)
    for (i,target) in enumerate([y_1,y_2,y_3,y_4]):
        if i ==0:
            target_name = 'identical'
        elif i == 1:
            target_name = 'phase_shifted'
        elif i == 2:
            target_name = 'Linear relation + complex noise'
        else:
            target_name = 'real noisy'
            
        mse_t = mse_time(x, target)
        mse_f = mse_freq_unitary_torch(x, target)  # equals unitary freq-domain MSE per sample
        amse   =    amse_loss(x, target) 

        print(f"{target_name:31s}  {mse_t.item():12.6f}    {mse_f.item():12.6f}   {amse.item():12.6f}  ")

Case                                 MSE_time       MSE_freq      AMSE_freq  
-------------------------------------------------------------------------------------
identical                            0.000000        0.000000       0.000000  
phase_shifted                        0.044662        0.044662       0.044662  
Linear relation + complex noise      0.262435        0.262435       0.616219  
real noisy                           0.010383        0.010383       0.030842  
